## Step 1: Install Required Packages

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless numpy torch torchvision
print("\n✓ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
from ultralytics import YOLO
import cv2
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
import threading
import imaplib
import email
import time
import uuid
import os
import sys
import traceback
import re
from google.colab.patches import cv2_imshow
from google.colab import files
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

print("✓ Libraries imported successfully!")

## Step 3: Configure Email Settings

**IMPORTANT:** For Gmail users:
1. Enable 2-Factor Authentication on your Google account
2. Generate an App Password: https://myaccount.google.com/apppasswords
3. Use the App Password (not your regular password) below

In [ ]:
# ============================================
# EMAIL CONFIGURATION - Set your credentials here
# ============================================
# For Gmail: Use App Password (not regular password)
# Generate at: https://myaccount.google.com/apppasswords
EMAIL_CONFIG = {
    'from_email': 'sabarideva05@gmail.com',
    'from_password': os.environ.get('ALERT_FROM_PASSWORD', ''),  # leave empty to prompt securely
    'to_email': 'sabithaekambaram1978@gmail.com',
    'followup_emails': 'dhevprince006@gmail.com,selvasvk2@gmail.com,johanalbert1015@gmail.com'
}
# ============================================
if not EMAIL_CONFIG['from_password']:
    from getpass import getpass
    EMAIL_CONFIG['from_password'] = getpass('Enter app password: ')

# Set environment variables from config
os.environ['ALERT_FROM_EMAIL'] = EMAIL_CONFIG['from_email']
os.environ['ALERT_FROM_PASSWORD'] = EMAIL_CONFIG['from_password']
os.environ['ALERT_TO_EMAIL'] = EMAIL_CONFIG['to_email']
os.environ['ALERT_FOLLOWUP_EMAILS'] = EMAIL_CONFIG['followup_emails']

print("✓ Email configuration set!")
print(f"  From: {EMAIL_CONFIG['from_email']}")
print(f"  To: {EMAIL_CONFIG['to_email']}")

## Step 4: Upload Video File

Upload the video file you want to analyze for person detection.

In [ ]:
# Upload video which is in ZIP file.
print("")
uploaded = files.upload()

video_filename = list(uploaded.keys())[0]
os.environ['VIDEO_PATH'] = video_filename

print(f"\n✓ Video uploaded: {video_filename}")

## Step 5: Download YOLO Model (if needed)

The YOLOv8 nano model will be automatically downloaded if not present.

In [ ]:
# Download YOLOv8 model
model_path = 'yolov8n.pt'

if not os.path.exists(model_path):
    print("Downloading YOLOv8 nano model...")
    model = YOLO('yolov8n.pt')  # This will auto-download
    print("✓ Model downloaded successfully!")
else:
    print("✓ Model already exists!")

os.environ['YOLO_MODEL_PATH'] = model_path

## Step 6: Define Helper Functions

These functions handle email alerts and reply monitoring.

In [ ]:
# Global variable to store alert snapshots
_ALERT_SNAPSHOTS = {}

def send_email_alert_with_image(subject, body, to_email, from_email, from_password, image_frame):
    """Send email alert with attached image."""
    msg = MIMEMultipart()
    msg["From"] = from_email
    msg["Reply-To"] = from_email
    msg["To"] = to_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, 'plain'))

    _, jpeg = cv2.imencode('.jpg', image_frame)
    image_bytes = jpeg.tobytes()
    image = MIMEImage(image_bytes, name="intruder.jpg")
    msg.attach(image)

    server = None
    try:
        server = smtplib.SMTP('smtp.gmail.com', 587, timeout=10)
        server.ehlo()
        server.starttls()
        server.ehlo()
        server.login(from_email, from_password)
        server.sendmail(from_email, to_email, msg.as_string())
        print('✓ Email alert with photo sent.')
    except Exception as e:
        print(f'✗ Failed to send email alert: {e}')
    finally:
        if server is not None:
            try:
                server.quit()
            except Exception:
                pass

def _imap_search_for_id(imap_conn, alert_id):
    """Search IMAP for messages containing alert_id."""
    try:
        imap_conn.select('INBOX')
        typ, data = imap_conn.search(None, 'ALL')
        if typ != 'OK':
            return []
        ids = data[0].split()
        if not ids:
            return []

        lookback = 100
        recent = ids[-lookback:]
        matches = []

        for mid in recent:
            try:
                typ2, msg_data = imap_conn.fetch(mid, '(RFC822)')
                if typ2 != 'OK' or not msg_data or not msg_data[0]:
                    continue
                raw = msg_data[0][1]
                if not raw:
                    continue
                needle = alert_id.encode(errors='ignore') if isinstance(alert_id, str) else alert_id
                if needle in raw:
                    matches.append(mid)
            except Exception:
                continue
        return matches
    except Exception:
        return []

def _extract_text_from_msg(msg_obj):
    """Extract text content from email message."""
    text_parts = []
    try:
        if msg_obj.is_multipart():
            for part in msg_obj.walk():
                ctype = part.get_content_type()
                disp = str(part.get('Content-Disposition'))
                if ctype == 'text/plain' and 'attachment' not in disp:
                    payload = part.get_payload(decode=True)
                    if payload:
                        text_parts.append(payload.decode(errors='ignore'))
                elif ctype == 'text/html' and 'attachment' not in disp:
                    payload = part.get_payload(decode=True)
                    if payload:
                        html = payload.decode(errors='ignore')
                        text = re.sub(r'<[^>]+>', '', html)
                        text_parts.append(text)
        else:
            payload = msg_obj.get_payload(decode=True)
            if payload:
                text_parts.append(payload.decode(errors='ignore'))
    except Exception as e:
        print(f"Error extracting text: {e}")
    return '\n'.join(text_parts)

def monitor_reply_and_forward(alert_id, from_email, from_password, primary_to, followup_list, timeout=120, poll_interval=5):
    """Monitor email replies and forward to security if person is unknown."""
    deadline = time.time() + timeout

    try:
        imap = imaplib.IMAP4_SSL('imap.gmail.com')
        imap.login(from_email, from_password)
    except Exception as e:
        print(f'✗ Failed to connect to IMAP: {e}')
        return

    try:
        while time.time() < deadline:
            ids = _imap_search_for_id(imap, alert_id)
            if ids:
                for mid in ids:
                    try:
                        typ, msg_data = imap.fetch(mid, '(RFC822)')
                        if typ != 'OK' or not msg_data or not msg_data[0]:
                            continue
                        raw = msg_data[0][1]
                        if not raw:
                            continue
                        msg = email.message_from_bytes(raw)

                        # Check if this is a reply to our alert (subject contains alert_id)
                        subject = msg.get('Subject', '')
                        if alert_id not in subject:
                            continue  # Skip emails not related to this alert

                        body = _extract_text_from_msg(msg)
                        body_lower = body.lower().strip()

                        # Check for explicit yes/no replies only
                        if body_lower.startswith('no') or '\nno\n' in body_lower or body_lower.endswith('no'):
                            print('⚠ Negative reply received - forwarding to security contacts')
                            snapshot = _ALERT_SNAPSHOTS.get(alert_id)
                            if snapshot is not None and followup_list:
                                for f in followup_list:
                                    try:
                                        send_email_alert_with_image(
                                            f'Intruder Alert (forwarded) [id: {alert_id}]',
                                            'Intruder confirmed unknown - forwarding snapshot.',
                                            f,
                                            from_email,
                                            from_password,
                                            snapshot
                                        )
                                    except Exception as e:
                                        print(f'✗ Error forwarding to {f}: {e}')
                                print('✓ Forwarding complete - terminating program.')
                            else:
                                print('No snapshot available to forward or no followup addresses configured.')
                            try:
                                imap.store(mid, '+FLAGS', r'\Seen') # Changed to raw string
                                imap.logout()
                            except Exception:
                                pass
                            os._exit(0)  # Force exit after forwarding
                            return
                        elif body_lower.startswith('yes') or '\nyes\n' in body_lower or body_lower.endswith('yes'):
                            print('✓ Positive reply received - person is known')
                            print('No forwarding needed - terminating program.')
                            try:
                                imap.store(mid, '+FLAGS', r'\Seen') # Changed to raw string
                                imap.logout()
                            except Exception:
                                pass
                            os._exit(0)  # Force exit on 'yes' reply - NO FORWARDING
                    except Exception as e:
                        print(f'Error processing message: {e}')
            time.sleep(poll_interval)
    finally:
        try:
            imap.logout()
        except Exception:
            pass

def handle_intruder_alert(image_frame):
    """Handle intruder detection by sending alert and monitoring replies."""
    from_email = os.environ.get('ALERT_FROM_EMAIL', '')
    from_password = os.environ.get('ALERT_FROM_PASSWORD', '')
    primary_to = os.environ.get('ALERT_TO_EMAIL', '')

    if not from_email or not from_password or not primary_to:
        print('⚠ WARNING: Email credentials not configured!')
        return

    followups_env = os.environ.get('ALERT_FOLLOWUP_EMAILS')
    followups = [e.strip() for e in followups_env.split(',') if e.strip()] if followups_env else []

    alert_id = str(uuid.uuid4())
    subject = f"Intruder Alert! [id: {alert_id}]"
    body = (
        "Intruder detected near your door!\n\n"
        "QUICK REPLY NEEDED:\n"
        "- Reply with 'yes' if you know this person\n"
        "- Reply with 'no' if you DON'T know this person (will forward to security contacts)\n\n"
        f"Alert ID: {alert_id}"
    )

    _ALERT_SNAPSHOTS[alert_id] = image_frame.copy()

    try:
        send_email_alert_with_image(subject, body, primary_to, from_email, from_password, image_frame)
        threading.Thread(
            target=monitor_reply_and_forward,
            args=(alert_id, from_email, from_password, primary_to, followups, 120, 1), # Changed poll_interval to 1 second
            daemon=True,
        ).start()
    except Exception as e:
        print(f'✗ Error handling alert: {e}')

def is_near_door(person_box, roi):
    """Check if person bounding box overlaps with door ROI."""
    px1, py1, px2, py2 = person_box
    rx1, ry1, rx2, ry2 = roi
    return not (px2 < rx1 or px1 > rx2 or py2 < ry1 or py1 > ry2)

print("✓ Helper functions defined!")

## Step 7: Run Person Detection

This cell processes the video and detects persons near the door.

In [ ]:
# Configuration
model_path = os.environ.get('YOLO_MODEL_PATH', 'yolov8n.pt')
video_path = os.environ.get('VIDEO_PATH')

frame_skip = 1  # Process every frame
frame_count = 0
alert_sent = False
DOOR_ROI = (50, 100, 250, 400)  # (x1, y1, x2, y2) - Adjust based on your video

# Load YOLO model
print("Loading YOLO model...")
model = YOLO(model_path)
print("✓ Model loaded successfully!")

# Open video
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"✗ Failed to open video: {video_path}")
else:
    print(f"✓ Video opened: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    print(f"  Total frames: {total_frames}")
    print(f"  FPS: {fps}")
    print("\nProcessing video...\n")

    # Process video
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % frame_skip != 0:
            continue

        if frame is None:
            continue

        # Resize for faster processing
        small_frame = cv2.resize(frame, (320, 320))
        results = model(small_frame, imgsz=320, conf=0.3)
        result = results[0]
        boxes = result.boxes

        scale_x = frame.shape[1] / 320
        scale_y = frame.shape[0] / 320

        # Check for person detections
        for box in boxes:
            cls = int(box.cls[0])
            if model.names[cls] == 'person':
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)
                conf = box.conf[0].item()
                label = f"{model.names[cls]} {conf:.2f}"

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # Check if person is near door
                if is_near_door((x1, y1, x2, y2), DOOR_ROI) and not alert_sent:
                    print(f'⚠ Person detected near door at frame {frame_count}!')
                    print('  Sending email alert...')
                    alert_sent = True

                    person_img = frame[y1:y2, x1:x2].copy()

                    # Send alert in background thread
                    threading.Thread(
                        target=handle_intruder_alert,
                        args=(person_img,),
                        daemon=True,
                    ).start()

                    # Display the detection
                    print("\nDetection snapshot:")
                    cv2_imshow(frame)
                    break

        # Progress update
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {frame_count}/{total_frames} frames ({progress:.1f}%)")

        # Break after alert is sent and detected
        if alert_sent:
            break

    cap.release()
    print("\n✓ Video processing complete!")

    if alert_sent:
        print("\n📧 Email alert sent! Check your inbox.")
        print("⏱ Monitoring for reply for 2 minutes...")
        print("   Reply 'yes' if you know the person")
        print("   Reply 'no' if you don't know the person (will forward to security)")
    else:
        print("\nℹ No person detected near the door.")

if alert_sent:
    print("Waiting for reply monitoring thread...")
    time.sleep(120)  # Wait 2 minutes for replies
    print("\n✓ Monitoring period complete!")
else:
    print("No alert was sent, skipping reply monitoring.")


## Notes

- Adjust `DOOR_ROI` coordinates in Step 7 to match your video's door location
- The default ROI is `(50, 100, 250, 400)` which represents (x1, y1, x2, y2) coordinates
- Modify `frame_skip` to process fewer frames for faster execution
- Adjust confidence threshold in the detection code (default: 0.3)

### Troubleshooting

- **Gmail authentication error**: Make sure you're using an App Password, not your regular password
- **Video not found**: Ensure the video was uploaded in Step 4
- **No detections**: Try lowering the confidence threshold or checking the DOOR_ROI coordinates
- **Email not received**: Check spam folder and verify email credentials